# 02 — Synthetic DGP Experiments

**Goal:** Understand *when* and *why* the panel estimators work by generating data under a fully controlled DGP.

Unlike the real-data notebook (01) and the semi-synthetic notebook (03), here every matrix is generated from scratch so $\tau^*$ is known exactly.

| Study | Question |
|:---|:---|
| **Step 1** | What do $M$, $Z$, $O$ look like? (DGP tour) |
| **Step 2** | How close is $\hat{\tau}$ to $\tau^*$ for each estimator? (single-run benchmark) |
| **Step 3** | Does error shrink as the panel grows in $N$ or $T$? (convergence) |
| **Step 4** | What happens when the true rank exceeds what methods assume? (rank misspecification) |
| **Step 5** | How does error scale with signal-to-noise ratio? (noise sensitivity) |

In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

from causaltensor.synthetic import generate, VALID_PATTERNS
from causaltensor.utils.common import get_tau_from_method

# Unicode labels used consistently in Plotly charts
ERR_LABEL  = "τ* = true ATT"
RERR_LABEL = "Mean relative error  |τ* − τ̂| / |τ*|"

ALL_METHODS  = ["DC_PR_auto_rank", "MC_NNM_CV", "CovariancePCA", "DID", "SDID", "SC", "RobustSyntheticControl"]
# MC_NNM_CV uses K-fold CV — slower; excluded from multi-trial sweeps by default
FAST_METHODS = ["DC_PR_auto_rank", "CovariancePCA", "DID", "SDID", "SC", "RobustSyntheticControl"]
PALETTE      = px.colors.qualitative.Plotly


def _run_methods(O, Z, methods=ALL_METHODS):
    """Return {method: tau_hat} for a single (O, Z) pair."""
    out = {}
    for m in methods:
        try:
            out[m] = float(get_tau_from_method(m, O, Z))
        except Exception:
            out[m] = np.nan
    return out


def sweep_param(param, values, *, N=50, T=80, rank=3, noise_scale=1.0,
                treatment_level=0.3, methods=FAST_METHODS, n_trials=5, seed=0):
    """
    Sweep one DGP parameter and record relative error per method.

    Parameters
    ----------
    param : str
        One of 'N', 'T', 'rank', 'noise_scale'.
    values : list
        Values to sweep.
    """
    records = []
    for val in values:
        # Build kwargs; override the swept parameter
        kwargs = dict(N=N, T=T, rank=rank, noise_scale=noise_scale,
                      treatment_pattern="Block", treatment_level=treatment_level)
        kwargs[param] = val

        for trial in range(n_trials):
            O_, Z_, tau_ = generate(seed=seed * 100 + trial, **kwargs)
            for m in methods:
                th = float(get_tau_from_method(m, O_, Z_))
                if not np.isnan(th) and tau_ != 0:
                    err = abs(tau_ - th) / abs(tau_)
                else:
                    err = np.nan
                records.append({param: val, "method": m, "trial": trial,
                                 "tau_star": tau_, "tau_hat": th, "error": err})
    return pd.DataFrame(records)

---
## Step 1 — DGP tour

The DGP follows a **low-rank factor model**:
$$M = UV^\top \;(\text{rank }r), \qquad O = M + \tau^*\cdot Z + \varepsilon$$
where $U \in \mathbb{R}^{N \times r}$, $V \in \mathbb{R}^{T \times r}$ are drawn i.i.d.~normal and $\varepsilon$ is additive Gaussian noise.

`generate(N, T, ...)` returns `(O, Z, tau_true)`.  
To visualise $M$ separately we call it again with `noise_scale=0` and `treatment_pattern=None` — since the RNG is identical (same `seed`), the baseline $M$ is exactly the same.

In [16]:
N, T, RANK = 50, 80, 3
SEED = 0

# Pure baseline (same M, no noise, no treatment)
M_clean, _, _ = generate(N, T, rank=RANK, noise_scale=0.0, treatment_pattern=None, seed=SEED)

# Observed panel with Block treatment
O, Z, tau_star = generate(N, T, rank=RANK, noise_scale=1.0,
                           treatment_pattern="Block", treatment_level=0.3, seed=SEED)

print(f"Panel  : {N} units x {T} periods,  rank = {RANK}")
print(f"tau*   : {tau_star:.4f}")
print(f"Treated: {int(Z.sum())} cells ({100*Z.mean():.1f}%)")

Panel  : 50 units x 80 periods,  rank = 3
tau*   : 0.3872
Treated: 225 cells (5.6%)


In [17]:
# Side-by-side: M (low-rank truth), Z (treatment mask), O (observed)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Low-rank baseline  M", "Treatment mask  Z", "Observed outcome  O"],
    horizontal_spacing=0.06,
)

vmax_M = float(np.percentile(np.abs(M_clean), 98))
fig.add_trace(go.Heatmap(z=M_clean, colorscale="RdBu_r", zmid=0,
                          zmin=-vmax_M, zmax=vmax_M, showscale=False), row=1, col=1)

fig.add_trace(go.Heatmap(z=Z.astype(float),
                          colorscale=[[0, "#e8e8e8"], [1, "#2980b9"]],
                          zmin=0, zmax=1, showscale=False), row=1, col=2)

vmax_O = float(np.percentile(np.abs(O), 98))
fig.add_trace(go.Heatmap(z=O, colorscale="RdBu_r", zmid=0,
                          zmin=-vmax_O, zmax=vmax_O, showscale=False), row=1, col=3)

fig.update_layout(
    title=dict(text=f"Synthetic panel DGP  (N={N}, T={T}, rank={RANK}, \u03c4*={tau_star:.3f})",
               x=0.5, xanchor="center"),
    height=400, margin=dict(l=40, r=40, t=80, b=40),
)
fig.show()

In [18]:
# All four treatment pattern shapes on the same panel dimensions
fig = make_subplots(rows=1, cols=4, subplot_titles=VALID_PATTERNS, horizontal_spacing=0.04)

for col, pat in enumerate(VALID_PATTERNS, start=1):
    _, Z_pat, _ = generate(N, T, rank=RANK, treatment_pattern=pat, seed=SEED + col)
    fig.add_trace(
        go.Heatmap(z=Z_pat.astype(float),
                   colorscale=[[0, "#e8e8e8"], [1, "#2980b9"]],
                   zmin=0, zmax=1, showscale=False),
        row=1, col=col)

fig.update_layout(
    title=dict(text="Treatment patterns  Z", x=0.5, xanchor="center"),
    height=360, margin=dict(l=40, r=40, t=60, b=40),
)
fig.show()

---
## Step 2 — Single-run benchmark

All seven estimators on the Block panel from Step 1. The dashed line is the ground truth $\tau^*$.

In [19]:
tau_hats = _run_methods(O, Z, methods=ALL_METHODS)

bench_df = pd.DataFrame({
    "method":  list(tau_hats.keys()),
    "tau_hat": list(tau_hats.values()),
})
bench_df["error"] = (bench_df["tau_hat"] - tau_star).abs() / abs(tau_star)
print(f"Ground truth tau* = {tau_star:.4f}")
display(bench_df.round(4))

Ground truth tau* = 0.3872


,method,tau_hat,error
0,DC_PR_auto_rank,0.3921,0.0126
1,MC_NNM_CV,0.3503,0.0954
2,CovariancePCA,0.3158,0.1845
3,DID,0.3621,0.0649
4,SDID,0.3513,0.0929
5,SC,0.2764,0.2861
6,RobustSyntheticControl,0.3097,0.2002


In [20]:
fig = go.Figure(go.Bar(
    x=bench_df["tau_hat"],
    y=bench_df["method"],
    orientation="h",
    marker_color=[PALETTE[i % len(PALETTE)] for i in range(len(bench_df))],
    text=bench_df["tau_hat"].round(3),
    textposition="outside",
    hovertemplate="%{y}: tau_hat=%{x:.4f}<extra></extra>",
))
fig.add_vline(x=tau_star, line_dash="dash", line_color="black", line_width=2,
              annotation_text=f"\u03c4* = {tau_star:.3f}",
              annotation_position="top right")
fig.update_layout(
    title=dict(text=f"Single-run ATT estimates vs. ground truth  (N={N}, T={T}, rank={RANK})",
               x=0.5, xanchor="center"),
    xaxis_title="\u03c4\u0302  (estimated ATT)",
    height=380,
    margin=dict(l=180, r=80, t=60, b=50),
    yaxis=dict(autorange="reversed"),
)
fig.show()

---
## Step 3 — Convergence as the panel grows

Theory predicts that estimation error shrinks as $N$ or $T$ grows (more data = better identification).  
We sweep each dimension independently, holding the other fixed, and plot mean relative error on a log $x$-axis.

Key predictions:
- **Matrix-completion methods** (DC-PR, MC-NNM) should converge in both $N$ and $T$.
- **DiD** benefits mainly from larger $T$ (more pre-treatment periods for trend estimation).
- **SC** benefits mainly from larger $N$ (more donor units for the synthetic control).

> **Runtime:** each cell runs ~90 s with `N_TRIALS=5`. Reduce to 3 for a quick check.

In [21]:
N_TRIALS = 5

print("Sweeping N (T=100 fixed)...")
df_N = sweep_param("N", [5, 15, 30, 75, 150, 300],
                   T=100, rank=3, noise_scale=1.0, n_trials=N_TRIALS)

print("Sweeping T (N=50 fixed)...")
df_T = sweep_param("T", [15, 40, 100, 250, 500],
                   N=50, rank=3, noise_scale=1.0, n_trials=N_TRIALS)

print("Done.")

Sweeping N (T=100 fixed)...
Sweeping T (N=50 fixed)...
Done.


In [28]:
def convergence_plot(df, param, xlabel, title):
    agg = df.groupby([param, "method"])["error"].mean().reset_index()
    fig = px.line(
        agg, x=param, y="error", color="method",
        markers=True,
        category_orders={"method": FAST_METHODS},
        labels={param: xlabel, "error": RERR_LABEL, "method": "Estimator"},
        title=title
    )
    fig.update_layout(
        height=440,
        margin=dict(l=60, r=20, t=60, b=110),
        legend=dict(orientation="h", y=-0.38, x=0.5, xanchor="center", title="Estimator"),
    )
    return fig

convergence_plot(df_N, "N", "Number of units  N  (log scale)",
                 "Convergence in N  (T=100, rank=3)").show()

convergence_plot(df_T, "T", "Number of periods  T  (log scale)",
                 "Convergence in T  (N=50, rank=3)").show()

---
## Step 4 — Rank misspecification

Every estimator implicitly assumes the panel has *low* rank $r$.  We generate data from a rank-$r$ model and sweep $r$ upward, keeping $N=50$, $T=100$ fixed.

Expected behaviour:
- **DC-PR / MC-NNM**: select rank adaptively — should degrade more gracefully.
- **DID / SDID**: do not exploit low rank — error driven by confounding structure, not rank per se.
- **SC / RSC**: rely on a good synthetic control match — high rank increases approximation error.

In [23]:
print("Sweeping rank...")
df_rank = sweep_param("rank", [1, 2, 3, 5, 8, 12, 20],
                      N=50, T=100, noise_scale=1.0, n_trials=N_TRIALS)
print("Done.")

Sweeping rank...
Done.


In [24]:
agg_rank = df_rank.groupby(["rank", "method"])["error"].mean().reset_index()

fig = px.line(
    agg_rank, x="rank", y="error", color="method",
    markers=True,
    category_orders={"method": FAST_METHODS},
    labels={"rank": "True rank  r", "error": RERR_LABEL, "method": "Estimator"},
    title="Rank misspecification  (N=50, T=100, noise=1)",
)
fig.update_layout(
    height=440,
    margin=dict(l=60, r=20, t=60, b=110),
    legend=dict(orientation="h", y=-0.38, x=0.5, xanchor="center", title="Estimator"),
)
fig.show()

---
## Step 5 — Noise sensitivity

$N$, $T$, rank are fixed; `noise_scale` $\sigma$ is swept from very low (high SNR) to very high (low SNR).
The injected signal is $\tau^* \approx 0.3 \cdot \text{mean}|M|$, so SNR $\approx \tau^*/\sigma$.

Expected: all methods degrade as $\sigma$ rises; matrix-completion methods should be more noise-resilient at intermediate SNR.

In [25]:
print("Sweeping noise_scale...")
df_noise = sweep_param("noise_scale", [0.05, 0.2, 0.5, 1.0, 2.0, 5.0],
                       N=50, T=100, rank=3, n_trials=N_TRIALS)
print("Done.")

tau_mean = df_noise["tau_star"].mean()
print(f"Mean tau* across trials: {tau_mean:.4f}")

Sweeping noise_scale...
Done.
Mean tau* across trials: 0.3756


In [27]:
agg_noise = df_noise.groupby(["noise_scale", "method"])["error"].mean().reset_index()

# Add approximate SNR as hover annotation
tau_mean = df_noise["tau_star"].mean()
agg_noise["SNR"] = (tau_mean / agg_noise["noise_scale"]).round(1)

fig = px.line(
    agg_noise, x="noise_scale", y="error", color="method",
    markers=True,
    category_orders={"method": FAST_METHODS},
    labels={"noise_scale": "Noise scale  \u03c3  (log scale)",
            "error": RERR_LABEL, "method": "Estimator"},
    title="Noise sensitivity  (N=50, T=100, rank=3)",
    hover_data={"SNR": True},
    # log_x=True,
)
fig.update_layout(
    height=440,
    margin=dict(l=60, r=20, t=60, b=110),
    legend=dict(orientation="h", y=-0.38, x=0.5, xanchor="center", title="Estimator"),
)
fig.show()

---
## Summary

| Finding | Detail |
|:---|:---|
| **Convergence** | All methods improve as $N$ or $T$ grows; the rate and dominant dimension vary by method. |
| **Rank robustness** | DC-PR and MC-NNM degrade more gracefully at high rank; SC and RSC are more sensitive. |
| **Noise** | Errors rise for all methods as $\sigma$ increases past $\tau^*$; low-rank methods maintain better SNR tolerance. |
| **Single-run variance** | Results can differ substantially across trials at small $N$ or $T$ — use multi-trial sweeps for reliable comparisons. |

### Next
- **`03_semi_synthetic_benchmarks.ipynb`** — combine real confounding with injected $\tau^*$ for the most realistic benchmark.